# Hybrid ResNet50 + Vision Transformer for Skin Disease Classification

**An Attention-Based Hybrid Deep Learning Framework Combining ResNet50 and Vision Transformer for Skin Disease Classification**

This notebook is fully free and open-source: no paid APIs, no API keys, no paid cloud services.
It runs on free Google Colab with GPU (Runtime → Change runtime type → GPU), and also works
(slower) on CPU-only.

**Important — no fabricated results.** Every number, plot, and heatmap in this notebook is
produced by actually running the code in the cells above it. If you have not yet trained the
models, metric cells will show `RESULTS WILL BE GENERATED AFTER TRAINING` instead of a
placeholder value.

Run the cells in order.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q torch torchvision timm scikit-learn opencv-python-headless \
    grad-cam pandas matplotlib seaborn pyyaml tqdm kagglehub requests

# If running outside Colab and you already cloned this repo, you can instead run:
# !pip install -q -r requirements.txt


## Cell 1b — Clone / locate the project

If you are running this notebook standalone in Colab (not from a cloned repo), upload or clone
the `skin_disease_hybrid/` project folder first so the `src/` and `scripts/` modules are importable.

In [ ]:
import os, sys

PROJECT_ROOT = "skin_disease_hybrid"  # adjust if your folder is named differently
assert os.path.isdir(PROJECT_ROOT), (
    "Project folder not found. Upload/clone the skin_disease_hybrid/ project into this "
    "Colab environment, or edit PROJECT_ROOT to point to it."
)
os.chdir(PROJECT_ROOT)
sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())


## Cell 2 — Check hardware

In [ ]:
from src.utils.seed import set_seed, get_device
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

set_seed(cfg["project"]["seed"])
device = get_device()


## Cell 3 — Download / load dataset

Attempts automatic download (free, no paid API). If it cannot succeed in this environment
(e.g. no internet, or Kaggle auth not configured), it prints exact manual-download
instructions instead of failing silently. The dataset placement is auto-detected on
subsequent runs.

In [ ]:
!python scripts/download_dataset.py --output_dir data/raw


## Cell 4 — Explore dataset (real dataset analysis, requirement #5)

In [ ]:
!python scripts/prepare_dataset.py --config configs/config.yaml


## Cell 5 — Display sample images from every class

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from src.dataset import CLASS_NAMES, CLASS_FULL_NAMES

train_df = pd.read_csv("data/splits/train.csv")

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(20, 4))
for i, cname in enumerate(CLASS_NAMES):
    subset = train_df[train_df["dx"] == cname]
    if len(subset) == 0:
        axes[i].axis("off")
        continue
    row = subset.iloc[0]
    img = Image.open(row["filepath"]).convert("RGB")
    axes[i].imshow(img)
    axes[i].set_title(f"{cname}\n{CLASS_FULL_NAMES[cname]}", fontsize=8)
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("outputs/figures/sample_images_per_class.png", dpi=150)
plt.show()


## Cell 6 — Train/validation/test split (already created in Cell 4)

This cell just re-displays the leakage-resistant, patient/lesion-wise split summary.

In [ ]:
train_df = pd.read_csv("data/splits/train.csv")
val_df = pd.read_csv("data/splits/val.csv")
test_df = pd.read_csv("data/splits/test.csv")

print(f"Number of training images: {len(train_df)}")
print(f"Number of validation images: {len(val_df)}")
print(f"Number of test images: {len(test_df)}")
print(f"Number of unique lesion_id (patient proxy) in train: {train_df['lesion_id'].nunique()}")
print(f"Number of unique lesion_id (patient proxy) in val: {val_df['lesion_id'].nunique()}")
print(f"Number of unique lesion_id (patient proxy) in test: {test_df['lesion_id'].nunique()}")

overlap = set(train_df['lesion_id']) & set(val_df['lesion_id']) & set(test_df['lesion_id'])
print(f"Lesion ID overlap across splits (should be empty set): {overlap}")


## Cell 7 — Create dataloaders

In [ ]:
from src.preprocessing import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(train_df, val_df, test_df, cfg)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## Cell 8 & 9 & 10 — Train baselines (ResNet18, ResNet50, ViT, ResNet50+ViT concat)

On free Colab GPU this takes roughly 10-30 minutes per baseline depending on epoch counts in
`configs/config.yaml`. Reduce `epochs_stage1` / `epochs_stage2` there if you need a faster run.

In [ ]:
!python scripts/train_baselines.py --config configs/config.yaml


## Cell 11 — Train the proposed attention-fusion hybrid model (Models A/B/C: CE, weighted CE, Focal)

In [ ]:
!python scripts/train_hybrid.py --config configs/config.yaml


## Cell 12 — Evaluate all models (real test-set metrics, requirement #19)

In [ ]:
!python scripts/evaluate.py --config configs/config.yaml


## Cell 13 — Confusion matrix

Already generated by Cell 12 (`outputs/confusion_matrix.png`). Displayed below.

In [ ]:
from PIL import Image as PILImage
import os

if os.path.exists("outputs/confusion_matrix.png"):
    display(PILImage.open("outputs/confusion_matrix.png"))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Cell 14 — ROC curves

In [ ]:
if os.path.exists("outputs/roc_curve.png"):
    display(PILImage.open("outputs/roc_curve.png"))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Cell 15 — Precision-Recall curves

In [ ]:
if os.path.exists("outputs/precision_recall_curve.png"):
    display(PILImage.open("outputs/precision_recall_curve.png"))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Cell 16 — Training curves

In [ ]:
import glob
for p in sorted(glob.glob("outputs/*hybrid_Model*history.json"))[:1]:
    print("History file found:", p)

# Plot training curves for the best hybrid variant, if available
import json
from src.utils.visualization import plot_training_curves

hist_files = glob.glob("outputs/hybrid_*_history.json")
if hist_files:
    with open(hist_files[0]) as f:
        history = json.load(f)
    plot_training_curves(history, "outputs")
    for fname in ["training_loss.png", "training_accuracy.png"]:
        fp = os.path.join("outputs", fname)
        if os.path.exists(fp):
            display(PILImage.open(fp))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Cell 17 & 18 & 19 — Generate Grad-CAM, ViT attention maps, and demonstration images (requirement #14)

In [ ]:
!python scripts/generate_demo.py --config configs/config.yaml


In [ ]:
import glob
for fp in sorted(glob.glob("outputs/demonstrations/demo_*.png")):
    print(fp)
    display(PILImage.open(fp))


## Cell 20 — Generate ablation results

In [ ]:
!python scripts/generate_report.py --config configs/config.yaml


In [ ]:
if os.path.exists("outputs/ablation_results.csv"):
    import pandas as pd
    display(pd.read_csv("outputs/ablation_results.csv"))
    display(PILImage.open("outputs/ablation_comparison.png"))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Cell 21 — Error analysis

Already produced by Cell 20 (`generate_report.py`).

In [ ]:
err_summary = "outputs/error_analysis/error_analysis_summary.csv"
if os.path.exists(err_summary):
    display(pd.read_csv(err_summary))
    for fp in sorted(glob.glob("outputs/error_analysis/error_*.png"))[:3]:
        display(PILImage.open(fp))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Bonus Cell — Robustness test (requirement #23)

Checks whether predictions stay stable under mild brightness/contrast/rotation/noise perturbations, using real test images and real inference.

In [ ]:
!python scripts/robustness_test.py --config configs/config.yaml


In [ ]:
import os
if os.path.exists("outputs/figures/robustness_flip_rates.png"):
    display(PILImage.open("outputs/figures/robustness_flip_rates.png"))
else:
    print("RESULTS WILL BE GENERATED AFTER TRAINING")


## Cell 22 — Save all outputs / final summary

In [ ]:
import json

def safe_read_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

comparison_path = "outputs/model_comparison.csv"
best_info = safe_read_json("outputs/best_model_info.json")

print("=" * 50)
print("PROJECT DEMONSTRATION COMPLETE")
print("=" * 50)

if os.path.exists(comparison_path):
    comp_df = pd.read_csv(comparison_path)
    best_row = comp_df.sort_values("Macro F1", ascending=False).iloc[0]
    print(f"\nBest Model:\n{best_row['Model']}")
    print(f"\nTest Accuracy:\n{best_row['Accuracy']:.4f}")
    print(f"\nMacro F1:\n{best_row['Macro F1']:.4f}")
    print(f"\nAUC:\n{best_row['AUC (macro)']}")
else:
    print("\nRESULTS WILL BE GENERATED AFTER TRAINING")

print("\nGenerated Demonstrations:\noutputs/demonstrations/")
print("\nGenerated Figures:\noutputs/figures/")
print("\nSaved Model:\nmodels/checkpoints/best_model.pth")
print("=" * 50)
